# 05. 딥 리서치 에이전트 — 병렬 서브에이전트와 5단계 워크플로

## 학습 목표

- 병렬 서브에이전트 3개(researcher-1, researcher-2, fact-checker)를 구성한다
- `think_tool`로 전략적 반성(strategic reflection)을 구현한다
- 5단계 워크플로(Plan → Delegate → Synthesize → Verify → Report)를 설계한다
- v1 미들웨어(SummarizationMiddleware, ModelCallLimitMiddleware, ModelFallbackMiddleware)를 적용한다


## 개요

| 항목 | 내용 |
|------|------|
| **프레임워크** | Deep Agents |
| **핵심 컴포넌트** | 병렬 서브에이전트 3개, think_tool |
| **워크플로** | 5단계: Plan → Delegate → Synthesize → Verify → Report |
| **백엔드** | `FilesystemBackend(root_dir=".", virtual_mode=True)` |
| **빌트인 도구** | `write_todos` (계획), `task` (서브에이전트 호출) |
| **스킬** | `skills/deep-research/SKILL.md` — 리서치 방법론 + 인용 규칙 |

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY를 .env에 설정하세요"


In [3]:
# Observability 설정 (선택)
import os

if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault(
        "LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default")
    )
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

In [2]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-5.4-mini")

/Users/jhj/Desktop/2026_1_sds_ax_advanced/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1단계: think_tool — 전략적 반성 도구

`think_tool`은 에이전트가 행동하기 전에 "생각"을 기록하는 도구입니다. 이 패턴은 에이전트의 의사결정 품질을 높입니다:

- 검색 결과를 분석하고 다음 행동을 계획
- 수집된 정보의 충분성을 평가
- 서브에이전트에게 위임할 작업을 구체화


In [ ]:
from langchain.tools import tool


# Self-Reflection 도구 정의
@tool
def think_tool(thought: str) -> str:
    """전략적 반성 — 현재 상황을 분석하고 다음 행동을 계획합니다."""
    return f"Reflection recorded: {thought}"


## 2단계: web_search 도구 (간소화)

실제 딥 리서치에서는 Tavily API를 사용하지만, 여기서는 학습 목적으로 간소화된 검색 도구를 정의합니다.


In [8]:
@tool
def web_search(query: str) -> str:
    """웹 검색을 수행합니다 (시뮬레이션)."""
    results = {
        "AI agent": "AI 에이전트는 자율적으로 작업을 수행하는 시스템입니다. 2024년 이후 급성장 중입니다.",
        "LangGraph": "LangGraph는 상태 기반 워크플로 프레임워크입니다. Graph API와 Functional API를 지원합니다.",
        "Deep Agents": "Deep Agents는 올인원 에이전트 SDK입니다. 서브에이전트, 백엔드, 스킬을 지원합니다.",
    }
    for key, val in results.items():
        if key.lower() in query.lower():
            return val
    return f"'{query}'에 대한 검색 결과: 관련 정보를 찾을 수 없습니다."


## 3단계: 5단계 리서치 워크플로 프롬프트

의 가 프롬프트를 로드합니다 (LangSmith Hub -> Langfuse -> 기본값).

| 단계 | 이름 | 설명 |
|------|------|------|
| 1 | **Plan** | 로 리서치 계획 작성 |
| 2 | **Delegate** | 서브에이전트에게 병렬 조사 위임 (최대 3개 동시) |
| 3 | **Synthesize** | 수집된 정보를 통합 |
| 4 | **Verify** | fact-checker가 사실 검증 |
| 5 | **Report** | 최종 보고서 작성 |

In [4]:
from prompts import RESEARCH_AGENT_PROMPT

print(RESEARCH_AGENT_PROMPT)

당신은 박사급 딥 리서치 에이전트입니다.

## 워크플로
1. **Plan**: write_todos로 리서치 계획을 세우세요
2. **Delegate**: 서브에이전트에게 조사를 위임하세요 (비교 분석 시 병렬)
3. **Synthesize**: 수집된 정보를 통합하세요
4. **Verify**: fact-checker에게 사실 검증을 요청하세요
5. **Report**: 최종 보고서를 작성하세요

## 규칙
- 검색 후 반드시 think_tool로 반성하세요
- 서브에이전트는 최대 3개까지 병렬 실행
- 인용은 [1], [2] 형식으로, 출처 섹션을 포함하세요
- 단순 주제는 서브에이전트 1개, 비교 분석은 2-3개 사용하세요


## 4단계: 서브에이전트 3개 정의

딥 리서치 에이전트는 3개의 전문 서브에이전트를 사용합니다:

| 서브에이전트 | 역할 | 도구 |
|------------|------|------|
| `researcher-1` | 주제 조사 담당 | web_search, think_tool |
| `researcher-2` | 비교/보완 조사 | web_search, think_tool |
| `fact-checker` | 사실 검증 담당 | web_search |


In [9]:
researcher_1 = {
    "name": "researcher-1",
    "description": "주제에 대한 심층 조사를 수행합니다",
    "system_prompt": "당신은 리서치 전문가입니다. 주제를 깊이 조사하고 핵심 정보를 요약하세요. 검색 후 think_tool로 반성하세요.",
    "tools": [web_search, think_tool],
}


In [10]:
researcher_2 = {
    "name": "researcher-2",
    "description": "보완적 관점에서 추가 조사를 수행합니다",
    "system_prompt": "당신은 보완 리서처입니다. 다른 관점에서 추가 정보를 수집하세요. 검색 후 think_tool로 반성하세요.",
    "tools": [web_search, think_tool],
}


In [11]:
fact_checker = {
    "name": "fact-checker",
    "description": "수집된 정보의 사실 여부를 검증합니다",
    "system_prompt": "당신은 팩트체커입니다. 제공된 정보의 정확성을 검증하고, 오류가 있으면 지적하세요.",
    "tools": [web_search],
}


## 5단계: 딥 리서치 에이전트 생성 (v1 미들웨어)

모든 도구와 서브에이전트를 조합하여 최종 에이전트를 생성합니다. v1 미들웨어로 안정성과 신뢰성을 강화합니다:

| 미들웨어 | 역할 |
|---------|------|
|  | 긴 리서치 대화를 자동 요약하여 컨텍스트 절약 |
|  | 리서치 무한 루프 방지 — 최대 30회 모델 호출 제한 |
|  | 주 모델 실패 시 백업 모델로 자동 전환 |

로 체크포인팅을 활성화하여 중단된 리서치를 재개할 수 있습니다.

In [12]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import (
    SummarizationMiddleware,
    ModelCallLimitMiddleware,
    ModelFallbackMiddleware,
)

research_agent = create_deep_agent(
    model=model,
    tools=[web_search, think_tool],
    subagents=[researcher_1, researcher_2, fact_checker],
    system_prompt=RESEARCH_AGENT_PROMPT,
    backend=FilesystemBackend(root_dir=".", virtual_mode=True),
    skills=["/skills/"],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(model=model, trigger=("messages", 15)),
        ModelCallLimitMiddleware(run_limit=30),
        ModelFallbackMiddleware("gpt-4.1-mini"),
    ],
)

## 6단계: 리서치 실행

에이전트에게 리서치 주제를 부여하면 5단계 워크플로를 자동으로 수행합니다.


In [14]:
thread = {"configurable": {"thread_id": "research-1"}}
response = research_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "AI 에이전트 프레임워크의 현재 동향을 조사해주세요. LangGraph와 Deep Agents를 중심으로 비교 분석해주세요.",
            }
        ]
    },
    config={**thread, **{}},
)
print(response["messages"][-1].content)

다음은 AI 에이전트 프레임워크의 현재 동향을 **LangGraph와 Deep Agents 중심**으로 비교 분석한 보고서입니다.

## 1) 한줄 요약

현재 AI 에이전트 프레임워크의 큰 흐름은 **“모델 호출 도구”에서 “상태·도구·반복·승인·복구를 다루는 실행 시스템”**으로 이동하고 있습니다.  
이 흐름에서 **LangGraph는 상태 기반 워크플로/그래프 오케스트레이션**에 강하고, **Deep Agents는 sub-agent와 skills를 활용한 작업 분해형 실행**에 강합니다.

---

## 2) 시장 동향 요약

2024년 이후 에이전트 프레임워크는 다음 방향으로 발전해 왔습니다.

- **상태(state) 관리 강화**
- **반복/분기/루프 같은 제어 흐름 강화**
- **체크포인팅과 재개(resume)**
- **human-in-the-loop 지원**
- **멀티에이전트 및 도구 사용 표준화**
- **장기 작업과 운영성(관찰 가능성, 복구성) 강화**

즉, 에이전트는 더 이상 “질문에 답하는 챗봇”이 아니라,  
**업무를 분해하고, 도구를 호출하고, 중간 상태를 저장하며, 필요하면 사람이 개입할 수 있는 실행 엔진**으로 진화하고 있습니다.

---

## 3) LangGraph 분석

### 핵심 포지션
LangGraph는 **상태 기반 그래프 프레임워크**입니다.  
에이전트 흐름을 노드와 엣지로 표현하되, 본질적으로는 **상태를 어떻게 갱신하고 다음 단계로 넘기느냐**를 중심에 둡니다.

### 최근 동향
1. **상태 중심 설계가 핵심 정체성**
   - 에이전트의 중간 상태, 분기 조건, 종료 조건을 명시적으로 다룸

2. **체크포인팅/복구/재개 강화**
   - 장시간 실행이나 실패 복구에 유리

3. **human-in-the-loop 패턴 지원**
   - 승인, 검토, 중단 후 재개가 중요한 업무에 적합

4. **멀티에이전트/서브그래프 활용**
   - 조사, 계획, 실행, 검증 같은 역할 분리에 유리

5

## 7단계: 스트리밍 — 네임스페이스 추적

`stream(subgraphs=True)`로 메인 에이전트와 서브에이전트의 실행 과정을 네임스페이스별로 추적합니다. 어떤 서브에이전트가 언제 호출되는지 실시간으로 확인할 수 있습니다.


In [15]:
# 네임스페이스별 청크 수 확인
from collections import Counter

thread2 = {"configurable": {"thread_id": "research-2"}}
chunks = []
for ns, chunk in research_agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "Deep Agents의 서브에이전트 아키텍처에 대해 조사해주세요.",
            }
        ]
    },
    config={**thread2, **{}},
    subgraphs=True,
):
    chunks.append((ns, chunk))

ns_counts = Counter(ns for ns, _ in chunks)
print(f"총 {len(chunks)}개 청크 수신", flush=True)
for ns_name, cnt in ns_counts.items():
    label = ns_name if ns_name else "main"
    print(f"  [{label}] {cnt}개 청크", flush=True)

총 145개 청크 수신
  [main] 33개 청크
  [('tools:2b28f050-3b4f-6a95-101b-706310888712',)] 32개 청크
  [('tools:56ed04b2-7188-947d-91ea-ccb05db33f74',)] 26개 청크
  [('tools:5680706a-f10b-ad8b-46a2-4ce9256fdd1c',)] 37개 청크
  [('tools:8cce5584-d7b1-7c6b-cbfb-452fca970f22',)] 17개 청크


## 서브에이전트 설계 모범 사례

| 원칙 | 설명 |
|------|------|
| **명확한 설명** | `description`을 구체적으로 작성 — 메인 에이전트가 위임 대상을 선택하는 기준 |
| **전문 프롬프트** | `system_prompt`에 출력 형식, 제약, 워크플로 포함 |
| **최소 도구** | 필요한 도구만 할당 — 불필요한 도구는 혼란 유발 |
| **간결한 결과** | 서브에이전트가 요약을 반환하도록 지시 — 원시 데이터 전달 금지 |


## 요약

| 항목 | 핵심 |
|------|------|
| **think_tool** | 전략적 반성 — 검색 후 분석, 다음 행동 계획 |
| **서브에이전트** | researcher-1, researcher-2, fact-checker 병렬 실행 |
| **워크플로** | Plan → Delegate → Synthesize → Verify → Report |
| **컨텍스트 관리** | 서브에이전트 결과만 메인에 전달 — 중간 과정 격리 |